**Swiggy Instamart Analytics, Triage & RAG Agent Project**

Part A & B: Customer Segmentation & RFM ClusteringIn this section, we load order and customer datasets, calculate RFM (Recency, Frequency, Monetary) metrics, and use K-Means clustering to identify high-value customer segments.
Code
**Snippet**
Python
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Load clean datasets
orders = pd.read_csv("data/clean/orders.csv")
products = pd.read_csv("data/clean/products.csv")

# Preprocess timestamps and amounts
orders["order_ts"] = pd.to_datetime(orders["order_ts"])
orders["order_amount"] = pd.to_numeric(orders["order_amount"], errors="coerce")
if "category" not in orders.columns:
    orders = orders.merge(products[["product_id", "category"]], on="product_id", how="left")

# Calculate RFM metrics
completed = orders[orders["status"] == "completed"].copy()
snapshot = pd.to_datetime("2025-10-01 00:00:00")
rfm = completed.groupby("customer_id").agg(
    recency=("order_ts", lambda x: int((snapshot - x.max()).days)),
    frequency=("order_id", "nunique"),
    monetary=("order_amount", lambda x: float(x.sum()))
).reset_index()

# K-Means Clustering (4 Segments)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm[["recency", "frequency", "monetary"]])
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X_scaled)
rfm["segment_id"] = kmeans.labels_

print(rfm.head(3))
Output PreviewPlaintext  customer_id  recency  frequency  monetary  segment_id
0      C0001        5         14   4520.00           1
1      C0002       45          2    540.50           0
2      C0003        2         22   8910.00           1


**Part C: Support Ticket Triage with Guardrails**


 we sanitize tickets by checking input length guardrails, applying strict PII regular expression redactions, running zero-shot and few-shot classification prompts, and managing retry loops with backoff.Code SnippetPythonimport re

tickets = pd.read_csv("data/raw/tickets.csv", keep_default_na=False)

email_pat = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
phone_pat = re.compile(r"\b[6-9]\d{9}\b")
order_pat = re.compile(r"ORD-\d{6}")

def redact_pii(text):
    txt = str(text)
    txt = email_pat.sub("[EMAIL]", txt)
    txt = phone_pat.sub("[PHONE]", txt)
    txt = order_pat.sub("[ORDER_ID]", txt)
    return txt

tickets["redacted"] = tickets["text"].apply(redact_pii)
print("Sample Original:", tickets["text"].iloc[0])
print("Sample Redacted:", tickets["redacted"].iloc[0])
Output PreviewPlaintextSample Original: Hi team, my email is john@gmail.com and order ORD-123456 was charged twice.
Sample Redacted: Hi team, my email is [EMAIL] and order [ORDER_ID] was charged twice.

**Part D: Policy RAG Assistant and Refund Agent**

In this section, we chunk policy text files, index them using TF-IDF vectorization, evaluate answers against ground truth queries, and simulate the ReAct agent tool-calling loop.Code SnippetPythonimport os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Simple policy chunking simulator
def get_chunks():
    return [
        {"source": "refund_policy.txt", "text": "Orders delivered in more than 45 minutes receive a 100% refund."},
        {"source": "escalation_policy.txt", "text": "Any refund request above INR 1000 must be escalated to a human supervisor."}
    ]

chunks = get_chunks()
vectorizer = TfidfVectorizer()
matrix = vectorizer.fit_transform([c["text"] for c in chunks])

query = "What happens if delivery takes over 45 minutes?"
query_vec = vectorizer.transform([query])
sims = cosine_similarity(query_vec, matrix).flatten()
best_idx = sims.argmax()

print("Top Matching Policy Source:", chunks[best_idx]["source"])
print("Matched Text Content:", chunks[best_idx]["text"])
Output PreviewPlaintextTop Matching Policy Source: refund_policy.txt
Matched Text Content: Orders delivered in more than 45 minutes receive a 100% refund.
